# T3 - Acoustic Propagation (standalone)
Run the cells top to bottom (VS Code: *Run All*). This module needs no other team's code: it sends a pulse from the 128-element array, lets it reflect off scatterers (mock tissue) and returns the pressure each receive element sees.

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
from nitk_usoundsim import config, acoustic_propagation as ap
print(f'elements {config.N_ELEMENTS} | f0 {config.F0/1e6:.1f} MHz | pitch {config.PITCH*1e3:.3f} mm | fs {config.FS/1e6:.0f} MHz | c {config.C:.0f} m/s')

## 1. Your scatterers
Each point is `x`, `z` in metres and an amplitude. Change them freely.

In [ ]:
pts = [dict(x=-3e-3, z=12e-3, amp=1.0), dict(x=0.0, z=22e-3, amp=0.8), dict(x=3e-3, z=32e-3, amp=0.6)]
res = ap.simulate_acoustic_propagation(pts)
print({k: getattr(v, 'shape', type(v).__name__) for k, v in res.items()})

## 2. The transmitted pulse and what one scatterer sees

In [ ]:
t = res['t_axis']*1e6
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
ax[0].plot(res['pulse_t']*1e6, res['pulse_p']); ax[0].set_title('TX pulse'); ax[0].set_xlabel('time [us]')
ax[1].plot(t, res['incident_pressure'][1]); ax[1].set_title('incident at scatterer 2'); ax[1].set_xlabel('time [us]')
ax[2].plot(t, res['reflected_pressure'][1], c='r'); ax[2].set_title('reflected at scatterer 2'); ax[2].set_xlabel('time [us]')
plt.tight_layout(); plt.show()

## 3. Output of this module: `pressure[element, time]`
This is the array handed to Receiver Beamforming.

In [ ]:
p = res['pressure']; v = np.percentile(np.abs(p), 99.5)
plt.figure(figsize=(7, 4)); plt.imshow(p.T, aspect='auto', cmap='seismic', vmin=-v, vmax=v, extent=[0, p.shape[0], t[-1], t[0]])
plt.xlabel('receive element'); plt.ylabel('time [us]'); plt.title(f'pressure {p.shape}'); plt.colorbar(); plt.show()

## 4. Attenuation vs depth and frequency

In [ ]:
z = np.linspace(0, 40e-3, 100)
for f in (4e6, 8e6, 12e6):
    plt.plot(z*1e3, ap.calculate_attenuation(z, f0=f), label=f'{f/1e6:.0f} MHz')
plt.xlabel('depth [mm]'); plt.ylabel('one-way amplitude factor'); plt.legend(); plt.show()